# Notebook 07 · Mini clasificador de sentimientos (end-to-end)

> **Unidad 2 · NLP e IA Generativa — Clase 1: Text Mining y Embeddings**  
> Ruta de la clase: 01 Texto → 02 Limpieza → 03 Tokenización → 04 Padding → 05 BoW/TF-IDF → 06 Embeddings → **07 Clasificador** → 08 Laboratorio

## Objetivo
Entrenar un clasificador binario (positivo/negativo) con **320 opiniones únicas**, aplicando todo el pipeline y **buenas prácticas**:

1. Dividir train/test **antes** de ajustar el tokenizer (evita *data leakage*).
2. Elegir `maxlen` con percentiles.
3. `mask_zero=True` para ignorar el padding.
4. `EarlyStopping` para no sobreajustar.
5. Evaluar con matriz de confusión y F1, no solo accuracy.
6. Analizar errores.

⏱️ Tiempo sugerido: 25 min

> 🧑‍🎓 **Versión para estudiantes.** Las celdas de código están vacías: complétalas en clase siguiendo las explicaciones.
> Cada celda trae **pistas** (funciones útiles) y los **nombres de variables** que deben quedar definidos, porque las celdas siguientes los usan.
> Ejecuta las celdas en orden. La versión resuelta está en la carpeta `notebooks/`.

In [ ]:
# ✅ Celda de configuración (ya resuelta): ejecútala antes de empezar.

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import layers
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

keras.utils.set_random_seed(42)
pd.set_option("display.max_colwidth", 70)

df = pd.read_csv("../datasets/opiniones_clientes.csv")
print("Shape:", df.shape, "| duplicados:", df["texto"].duplicated().sum())
print(df["sentimiento"].value_counts())
display(df.head())

## 1) Limpieza conservadora
Conservamos tildes, `ñ` y la palabra **`no`**: son importantes para el sentimiento.

In [ ]:
# ✍️ Ejercicio 1: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): apply, lower, sub, strip, head
# 📌 Al terminar deberían existir: limpiar_texto(), df['texto_limpio']


## 2) Split **antes** de tokenizar
Si ajustamos el tokenizer con todo el dataset, el vocabulario "conoce" palabras del test: eso es una pequeña **fuga de información**. En producción el modelo nunca conocerá las palabras futuras.

In [ ]:
# ✍️ Ejercicio 2: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): train_test_split, astype
# 📌 Al terminar deberían existir: y, txt_train, txt_test, y_train, y_test


## 3) Tokenización (solo con train) y tasa de OOV

In [ ]:
# ✍️ Ejercicio 3: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): Tokenizer, fit_on_texts, texts_to_sequences, mean
# 📌 Al terminar deberían existir: tokenizer, seq_train, seq_test, oov_rate


## 4) Padding con `maxlen` basado en datos

In [ ]:
# ✍️ Ejercicio 4: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): pad_sequences, ceil, percentile
# 📌 Al terminar deberían existir: longitudes, MAX_LEN, X_train, X_test


## 5) Modelo
`Embedding(mask_zero=True) → GlobalAveragePooling1D → Dropout → Dense(ReLU) → Dense(Sigmoid)`

In [ ]:
# ✍️ Ejercicio 5: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): Sequential, compile, summary, Input, Embedding, GlobalAveragePooling1D, Dropout, Dense
# 📌 Al terminar deberían existir: VOCAB, EMB_DIM, model


## 6) Entrenamiento con EarlyStopping

In [ ]:
# ✍️ Ejercicio 6: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): EarlyStopping, fit, subplots, tight_layout, show, plot, set_title, set_xlabel
# 📌 Al terminar deberían existir: early, history, fig, axes


## 7) Evaluación en test

In [ ]:
# ✍️ Ejercicio 7: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): ravel, astype, from_predictions, title, show, classification_report, predict
# 📌 Al terminar deberían existir: prob_test, pred_test


## 8) Análisis de errores
Las métricas dicen **cuánto** falla el modelo; leer los errores dice **por qué**.

In [ ]:
# ✍️ Ejercicio 8: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): DataFrame, sequences_to_texts, sort_values
# 📌 Al terminar deberían existir: resultados, resultados['texto_que_ve_el_modelo'], errores


## 9) Prueba tus propias frases
Modifica la lista y vuelve a ejecutar. Prueba **negaciones**, sarcasmo y palabras nuevas.

In [ ]:
# ✍️ Ejercicio 9: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): texts_to_sequences, pad_sequences, limpiar_texto, predict
# 📌 Al terminar deberían existir: predecir_sentimiento(), mis_frases


## Preguntas de reflexión
1. ¿El modelo comete más falsos positivos o falsos negativos? ¿Cuál es más costoso para el negocio?
2. ¿Por qué falla con *"no estuvo nada mal"* o con el sarcasmo? ¿Qué le falta al promedio de embeddings? (pista: **orden y contexto**).
3. ¿Cómo cambian los resultados al modificar `MAX_LEN` o `EMB_DIM`?
4. ¿Qué limitaciones tiene entrenar con solo 320 ejemplos? Compara con los 50.000 del proyecto IMDb.